# 4. Classification thématique : texte seul face aux GNN

Multi-étiquettes sur les 10 descripteurs EuroVoc présents dans au moins 10 actes. Validation GroupKFold à 5 plis par acte. Résultats complets lus dans `results/classification.json` (`graphrag/classify.py`), puis pli 0 rejoué en direct pour 5 modèles.

_Notebook rejoué avec le code du projet (`graphrag/`, et `app/` du projet vectoriel). Ses sorties sont réelles : il a été exécuté le 10/11/2026 à 03:04 par `scripts/build_notebooks.py`. Aucun appel au LLM n'est fait ici ; Neo4j doit être démarré._

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# candidats de recherche déjà calculés (MultiQuery, HyDE) : aucune requête au LLM
os.environ.setdefault("SEED_CACHE_PATH", str(ROOT / "data" / "cache" / "user_test_candidates.json"))

import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

from graphrag import config
config.use_vector_rag()
DATA = config.VECTOR_RAG_DIR / "data" / "eurlex"
RESULTS = ROOT / "results"
print("Projet :", ROOT)
print("Données du projet vectoriel :", DATA)

Projet : C:\Users\hazem\5eme\GraphRAG-juridique
Données du projet vectoriel : C:\Users\hazem\5eme\Advenced-Rag-Agents-main\Advenced-Rag-Agents-main\data\eurlex


In [2]:
clf = json.loads((RESULTS / "classification.json").read_text(encoding="utf-8"))
print("Étiquettes :", clf["labels"])
print("Configuration :", clf["config"])
tableau = pd.DataFrame({m: {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in s.items()} for m, s in clf["summary"].items()}).T
tableau.sort_values("macro_ap", ascending=False)

Étiquettes : ['Internet', 'données personnelles', 'harmonisation des normes', 'marché unique', 'marché unique numérique', 'prestation de services', 'protection des données', 'sécurité des systèmes d’information', 'technologie numérique', "échange d'information"]
Configuration : {'hidden': 256, 'heads': 4, 'dropout': 0.3, 'lr': 0.002, 'max_epochs': 300, 'patience': 30}


,micro_f1,macro_f1,macro_ap,act_micro_f1
logreg_act_mean,0.432 ± 0.139,0.387 ± 0.128,0.537 ± 0.162,0.446 ± 0.125
logreg,0.370 ± 0.089,0.320 ± 0.073,0.420 ± 0.128,0.419 ± 0.092
gt_cross,0.352 ± 0.083,0.294 ± 0.066,0.357 ± 0.104,0.416 ± 0.115
gat_all,0.326 ± 0.067,0.285 ± 0.099,0.343 ± 0.133,0.383 ± 0.052
gt_all,0.275 ± 0.066,0.214 ± 0.082,0.273 ± 0.093,0.352 ± 0.046
gat_cross,0.299 ± 0.056,0.262 ± 0.088,0.253 ± 0.105,0.366 ± 0.026
gcn_cross,0.275 ± 0.066,0.231 ± 0.080,0.208 ± 0.059,0.347 ± 0.063
mlp,0.275 ± 0.064,0.238 ± 0.072,0.208 ± 0.052,0.355 ± 0.047
gcn_all,0.269 ± 0.068,0.234 ± 0.072,0.170 ± 0.054,0.350 ± 0.052


## Les trois GNN (écrits en PyTorch, torch_geometric n'étant pas installable)

In [3]:
from graphrag import classify as C

print(C.GCN(768, 10))
print(C.GAT(768, 10))
print(C.GraphTransformer(768, 10))

GCN(
  (input): Linear(in_features=768, out_features=256, bias=True)
  (l1): Linear(in_features=256, out_features=256, bias=True)
  (l2): Linear(in_features=256, out_features=10, bias=True)
)
GAT(
  (input): Linear(in_features=768, out_features=256, bias=True)
  (l1): GATLayer(
    (proj): Linear(in_features=256, out_features=256, bias=False)
  )
  (l2): GATLayer(
    (proj): Linear(in_features=256, out_features=256, bias=False)
  )
  (head): Linear(in_features=256, out_features=10, bias=True)
)
GraphTransformer(
  (input): Linear(in_features=776, out_features=256, bias=True)
  (l1): GTLayer(
    (q): Linear(in_features=256, out_features=256, bias=True)
    (k): Linear(in_features=256, out_features=256, bias=True)
    (v): Linear(in_features=256, out_features=256, bias=True)
    (out): Linear(in_features=256, out_features=256, bias=True)
    (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=

## Pli 0 rejoué

Mêmes données, même découpage, même graine que le script : on vérifie que les chiffres enregistrés se reproduisent.

In [4]:
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

ids, celex, x, y, labels, edges = C.load_data()
graphs = {name: C.build_graph(pairs, len(ids)) for name, pairs in edges.items()}
train_all, test = next(GroupKFold(n_splits=5).split(x, y, groups=celex))
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=C.SEED).split(train_all, groups=celex[train_all]))
train, val = train_all[tr], train_all[va]
print(len(train), "articles d'entraînement,", len(val), "de validation,", len(test), "de test ;",
      "actes de test vus à l'entraînement :", len(set(celex[test]) & set(celex[train_all])))

2765 articles d'entraînement, 387 de validation, 789 de test ; actes de test vus à l'entraînement : 0


In [5]:
lignes = []
for name in ["logreg", "mlp", "gcn_all", "gat_all", "gt_cross"]:
    if name == "logreg":
        probs = C.fit_logreg(x, y, train, val)
    else:
        kind, _, edge_set = name.partition("_")
        probs, _ = C.fit_neural(kind, x, y, graphs[edge_set or "all"], train, val, C.SEED, C.MAX_EPOCHS)
    res = C.scores(y[test], probs[test], celex[test], C.best_thresholds(y[val], probs[val]))
    lignes.append({"modele": name, "macro_AP_rejoue": round(res["macro_ap"], 3),
                   "macro_AP_enregistre": round(clf["per_fold"][name][0]["macro_ap"], 3),
                   "micro_F1_rejoue": round(res["micro_f1"], 3)})
pd.DataFrame(lignes)

,modele,macro_AP_rejoue,macro_AP_enregistre,micro_F1_rejoue
0,logreg,0.259,0.259,0.205
1,mlp,0.185,0.185,0.224
2,gcn_all,0.109,0.109,0.192
3,gat_all,0.312,0.312,0.248
4,gt_cross,0.242,0.242,0.288


**Lecture :** le meilleur modèle n'utilise aucune arête (régression logistique + moyenne de l'acte). À entraînement identique, Graph Transformer et GAT dépassent le MLP ; le GCN non. Hypothèse non testée : les étiquettes décrivent l'acte et 91 % des renvois restent dans l'acte.